In [691]:
#!pip install selenium
#!pip install dotenv
#!pip install beautifulsoup4
!pip install pandas

In [692]:
import os #manipulando documentos do diretorio (.env)
from dotenv import load_dotenv #carrega a .env
from selenium import webdriver #driver do seleniu
from selenium.webdriver import ActionChains #Para executar click e escrita como mouse e teclado
from selenium.webdriver.common.by import By #Para identificar os tipos de elementos
from selenium.webdriver.support.ui import WebDriverWait #para criar o tempo de espera para interacoes
from selenium.webdriver.support import expected_conditions as EC #condicao de espera para a interacao

import time #tempo de carregar a pagina para o BS4 ler

from bs4 import BeautifulSoup #para ler o html desejado

import re #para tratar os dados capturados

import pandas as pd #para criacao e analise do dataframe

In [693]:
#Carrega variaveis secretas
load_dotenv()

#Armazena variaveis secretas
usuario = os.getenv("USER_NAME")
password = os.getenv("USER_PASSWORD")
tWait = float(os.getenv("T_WAIT")) #tempo de espera para carregar o .html da pagina, use uma que encaixe com a sua maquina, teste e erro GERALMENTE COM 10 DA CERTO

In [694]:
#Acessa o site RHonline pelo chrome
driver = webdriver.Chrome()
driver.get("https://rhonline.msgas.com.br/#/login")

#CRIANDO ACTION PARA PREENCHER CAMPOS E CLICAR
#site precisa do action para funcionar pois utilizando o driver puro nao foi capaz de escrever
#   estava retornando erro: ElementNotInteractableException
action = ActionChains(driver)

#UTILIZA METODO EXPLICIT WAIT
wait = WebDriverWait(driver, 10)#5 segundos de espera

In [695]:
#--------------------------
#
#   LOGIN NA PAGINA
#
#--------------------------

In [696]:
#identificacao dos campos de interacao
txtUser = wait.until(EC.element_to_be_clickable((By.NAME, "user"))) #campo do usuario
txtPassword = wait.until(EC.element_to_be_clickable((By.NAME, "password"))) #campo de senha
btnEntrar = wait.until(EC.visibility_of_element_located((By.TAG_NAME, "button")))

#Escreve o usuario e senha
action.move_to_element(txtUser).click().send_keys(usuario).perform()
action.move_to_element(txtPassword).click().send_keys(password).perform()

#clica em login
btnEntrar.click()

In [697]:
#--------------------------------------
#
#       Navega ate pagina de pontos
#
#--------------------------------------

In [698]:
#NAVEGAR PARA PAGINA DE PONTO
#   "botao" para clicar-> <span _ngcontent-ng-c3029837786="" aria-label="Acessar menu:  Ponto" class="an an-clock"></span>
ponto = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[contains(@aria-label, 'Ponto')]")))
ponto.click()

#NAVEGA PARA O ESPELHO DE PONTO
#   "botao" para clicar -> <p _ngcontent-ng-c298496229=""> Espelho de ponto </p>
espelho = wait.until(EC.element_to_be_clickable((By.XPATH, "//p[contains(text(), 'Espelho de ponto')]")))
espelho.click()

In [699]:
#----------------------------------------------------------------
#
#   Extrai os dados dos pontos (daquele periodo)
#       (mas para esse teste utilizarei o periodo anterior)
#
#---------------------------------------------------------------

In [700]:
#!!!!!!!!!!aba para ser deletada, apenas para o teste anterior!!!!!!!!!

# opcao para clicar-> <option value="{&quot;initDate&quot;:&quot;2026-04-16T12:00:00Z&quot;,&quot;endDate&quot;:&quot;2026-05-15T12:00:00Z&quot;,&quot;actualPeriod&quot;:true,&quot;id&quot;:&quot;&quot;}"> 16/04/2026 - 15/05/2026 </option>
data_analisada = wait.until(EC.element_to_be_clickable((By.XPATH, "//option[contains(text(), '16/03/2026')]")))
data_analisada.click()

In [701]:
#HORA DO SHOWWW BEATIFULLSOAPPPP
#espera a pagina carregar plenamento para entao extrair o .html
time.sleep(tWait)

page = driver.page_source
soup = BeautifulSoup(page, 'html.parser')
driver.quit() #nao sera mais necessario navegacao

In [702]:
#!!!!!!!!aba [ara ser deletada, apenas para analisar o html mais facilmente
with open('teste.html', 'w', encoding='utf-8') as f:
    f.write(page)

In [703]:
#------------------------------------
#
#   TRATAR OS DADOS DO HTML
#
#------------------------------------

In [704]:
#Tratar os dados do html
#Cada linha(tr) possui os dados dos pontos no text, com sujeira junto ;-;
#HORA DA FAXINA! vamos limpar os dados
#   INFORMACOES PARA CRIACAO DO DATAFRAME
#       DIA DO PONTO : ENTRADA 1, SAIDA 1, ENTRADA 2, SAIDA 2
lines = soup.tbody.find_all('tr')
lines[0].get_text(separator=" ") # exemplo de 1 saida, separador para melhorar a visibilidade

'15/04 quarta-feira qua.  07:25   Entrada 1   09:45   Saída 1   13:30   Entrada 2   17:05   Saída 2  Resumo diário Incluir batida Solicitar abono Enviar atestado'

In [718]:
#Criacao do dataframe com o loop
data = []
for line in lines:
    text_line = line.get_text(separator=" ")

    #pega a data com o dia
    match_day = re.search(r"(\d{2}/\d{2})\s*([a-z-áéíóúç.]+)", text_line, re.IGNORECASE)

    print(match_day)

    if match_day: #se realmente existir dia
        schedule_day = match_day.group(1) #dia do calendario ex: 01/01
        day = match_day.group(2) #dia ex: segunda-feira

        hours = re.findall(r"(\d{2}:\d{2})", text_line)

        item = {"Data": schedule_day, "Dia": day}#linha do dataframe

        #nomeia as batidas de ponto
        for i in range(len(hours)):
            col_name = f"Batida_{i+1}"
            item[col_name] = hours[i]

        data.append(item)

<re.Match object; span=(0, 18), match='15/04 quarta-feira'>
<re.Match object; span=(0, 17), match='14/04 terça-feira'>
<re.Match object; span=(0, 19), match='13/04 segunda-feira'>
<re.Match object; span=(0, 13), match='12/04 domingo'>
<re.Match object; span=(0, 12), match='11/04 sábado'>
<re.Match object; span=(0, 17), match='10/04 sexta-feira'>
<re.Match object; span=(0, 18), match='09/04 quinta-feira'>
<re.Match object; span=(0, 18), match='08/04 quarta-feira'>
<re.Match object; span=(0, 17), match='07/04 terça-feira'>
<re.Match object; span=(0, 19), match='06/04 segunda-feira'>
<re.Match object; span=(0, 13), match='05/04 domingo'>
<re.Match object; span=(0, 12), match='04/04 sábado'>
<re.Match object; span=(0, 17), match='03/04 sexta-feira'>
<re.Match object; span=(0, 18), match='02/04 quinta-feira'>
<re.Match object; span=(0, 18), match='01/04 quarta-feira'>
<re.Match object; span=(0, 17), match='31/03 terça-feira'>
<re.Match object; span=(0, 19), match='30/03 segunda-feira'>
<re.

In [723]:
df = pd.DataFrame(data)